<a href="https://colab.research.google.com/github/jyoti-ai-generalist-portfolio/AIHorizon/blob/main/Proj3_AI_Lead_Qualifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Project 3: AI Lead Qualifier

PROJECT: Input lead description → classify industry, budget range, intent, urgency and recommended next action → output structured JSON + CSV.

In [16]:
import json
from google.colab import userdata
from google import genai
from google.genai import types
from pydantic import BaseModel

import pandas as pd
import csv



class LeadClassification(BaseModel):
  LeadId: str
  industry: str
  budget_range: str
  intent: str
  urgency: str
  recommended_next_action: str


try:

  #df = pd.read_csv('crm_sample_leads.csv')
  #print(df.head())
  csv_file_path =  'crm_sample_leads.csv'

  """with open(csv_file_path, mode='r', encoding='utf-8') as csv_file:
    # DictReader automatically treats the first row as headers/keys
    csv_reader = csv.DictReader(csv_file)
    data = list(csv_reader)
    # data is of type dictionary
    print (data)
  """

  client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
  myprompt = """The given file has lead descriptions for CRM.
  Based on lead description classify the Industry, Budget Range,Intent,
  Urgency and Recommended Next Action against each Lead ID.
  Return the response as a JSON
  """

  csv_file = client.files.upload(file=csv_file_path)
  response = client.models.generate_content( \
    model="gemini-3.5-flash-lite", \
    contents=[csv_file,myprompt], \
    config=types.GenerateContentConfig( \
        response_mime_type="application/json", \
        response_schema=list[LeadClassification],  # Pass the Pydantic class here \
    ))

  #pd.read_json(response.text)
  output_file_path = "output_lead_actions.csv"
  myPythonObj = json.loads(response.text)  # Parses to a list of dicts

  # Convert list of dicts to DataFrame
  df = pd.DataFrame(myPythonObj)
  df.to_csv(output_file_path, index=False)
  print ("Check output file for the details ", output_file_path)
except FileNotFoundError:
  print ("Input file not found")
except json.JSONDecodeError:
  print ("File not in JSON Format")







Check output file for the details  output_lead_actions.csv
